# Módulo 7 · Clase 7 — Cross-Validation y GridSearchCV

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Tema |
|---|------|
| 1 | Por qué necesitamos CV: el problema del test set único |
| 2 | K-Fold Cross-Validation y variantes |
| 3 | `cross_val_score` |
| 4 | GridSearchCV — búsqueda exhaustiva |
| 5 | RandomizedSearchCV — búsqueda eficiente |
| 6 | Pipeline + GridSearchCV |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold,
    cross_val_score, GridSearchCV, RandomizedSearchCV
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import make_classification, load_breast_cancer
from sklearn.metrics import accuracy_score

np.random.seed(42)
print("Librerías listas")

---
## 1. El problema del test set único

```
Evaluación ingenua:
  Dividir datos → entrenar → evaluar en test → reportar el score
  Problema: el score depende de QUÉ datos cayeron en test.
  Con pocas muestras, la varianza del score puede ser muy alta.

Ejemplo:
  100 muestras, test_size=0.2 → 20 muestras de test
  Un modelo puede tener accuracy 0.75 o 0.90 solo por azar
  en función de qué 20 muestras cayeron en test.

Solución: Cross-Validation
  Usa TODOS los datos tanto para entrenar como para validar,
  en turnos distintos. Da un score más fiable y una estimación
  de su incertidumbre (desviación típica).
```

In [ ]:
from sklearn.datasets import load_breast_cancer

X, y = load_breast_cancer(return_X_y=True)
print(f"Dataset: {X.shape[0]} muestras, {X.shape[1]} features")

# Demostrar la variabilidad del score con un único test set
modelo = DecisionTreeClassifier(max_depth=4, random_state=None)  # sin semilla fija

scores_aleatorios = []
for i in range(30):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2)
    scores_aleatorios.append(accuracy_score(y_te, modelo.fit(X_tr, y_tr).predict(X_te)))

print(f"\n30 evaluaciones con test set aleatorio:")
print(f"  Mínimo: {min(scores_aleatorios):.4f}")
print(f"  Máximo: {max(scores_aleatorios):.4f}")
print(f"  Rango:  {max(scores_aleatorios)-min(scores_aleatorios):.4f}  ← ¡muy variable!")

plt.figure(figsize=(8, 3))
plt.plot(range(30), scores_aleatorios, 'bo-', ms=5, alpha=0.7)
plt.axhline(np.mean(scores_aleatorios), color='red', lw=2, ls='--', label=f'Media={np.mean(scores_aleatorios):.4f}')
plt.fill_between(range(30), np.mean(scores_aleatorios)-np.std(scores_aleatorios),
                 np.mean(scores_aleatorios)+np.std(scores_aleatorios), alpha=0.2, color='red')
plt.xlabel('Experimento'); plt.ylabel('Accuracy')
plt.title('Variabilidad del score con test set aleatorio')
plt.legend(); plt.tight_layout(); plt.show()

---
## 2. K-Fold Cross-Validation

```
Algoritmo K-Fold (K=5 típico):

  Datos completos: [████████████████████] (100 muestras)

  Fold 1: [Test][Train][Train][Train][Train]
  Fold 2: [Train][Test][Train][Train][Train]
  Fold 3: [Train][Train][Test][Train][Train]
  Fold 4: [Train][Train][Train][Test][Train]
  Fold 5: [Train][Train][Train][Train][Test]

  → 5 modelos entrenados, 5 scores
  → score_final = media(5 scores) ± std(5 scores)

Variantes:
  StratifiedKFold  → mantiene la proporción de clases en cada fold
                     (SIEMPRE usar en clasificación)
  LeaveOneOut      → K = n_muestras (muy caro, pequeños datasets)
  RepeatedKFold    → repite K-Fold varias veces con distinta semilla
```

In [ ]:
# Visualizar los folds
X_demo = np.arange(10).reshape(-1, 1)
y_demo = np.array([0,1,0,1,0,1,0,1,0,1])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, (nombre, cv) in zip(axes, [
    ('KFold (K=5)', KFold(n_splits=5, shuffle=True, random_state=42)),
    ('StratifiedKFold (K=5)', StratifiedKFold(n_splits=5, shuffle=True, random_state=42)),
]):
    grid = np.zeros((5, 10))
    for fold_idx, (tr_idx, te_idx) in enumerate(cv.split(X_demo, y_demo)):
        grid[fold_idx, tr_idx] = 1
        grid[fold_idx, te_idx] = 2

    ax.imshow(grid, cmap=plt.cm.get_cmap('RdYlGn', 3), aspect='auto', vmin=0, vmax=2)
    ax.set_xticks(range(10))
    ax.set_xticklabels([f's{i}\n(y={y_demo[i]})' for i in range(10)], fontsize=8)
    ax.set_yticks(range(5))
    ax.set_yticklabels([f'Fold {i+1}' for i in range(5)])
    ax.set_title(nombre)

from matplotlib.patches import Patch
legend = [Patch(facecolor='red', label='Test'), Patch(facecolor='green', label='Train')]
axes[0].legend(handles=legend, loc='lower right', fontsize=8)
plt.suptitle('Estructura de los folds (verde=train, rojo=test)', fontsize=10)
plt.tight_layout(); plt.show()

---
## 3. `cross_val_score`

In [ ]:
X, y = load_breast_cancer(return_X_y=True)

modelos = [
    ('Árbol depth=1', DecisionTreeClassifier(max_depth=1, random_state=42)),
    ('Árbol depth=5', DecisionTreeClassifier(max_depth=5, random_state=42)),
    ('Random Forest', RandomForestClassifier(100, random_state=42, n_jobs=-1)),
    ('LogisticReg',   LogisticRegression(max_iter=5000, random_state=42)),
]

print(f"{'Modelo':20} {'Acc CV (media±std)':22} {'Min':8} {'Max':8}")
print("-" * 65)

resultados = {}
for nombre, m in modelos:
    scores = cross_val_score(
        m, X, y,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring='accuracy',
        n_jobs=-1
    )
    resultados[nombre] = scores
    print(f"{nombre:20} {scores.mean():.4f} ± {scores.std():.4f}         "
          f"{scores.min():.4f}  {scores.max():.4f}")

# Visualizar
fig, ax = plt.subplots(figsize=(9, 4))
nombres = list(resultados.keys())
datos   = list(resultados.values())
ax.boxplot(datos, labels=nombres, patch_artist=True,
           boxprops=dict(facecolor='lightblue'))
ax.set_ylabel('Accuracy')
ax.set_title('Distribución de scores — 5-Fold CV')
ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

---
## 4. GridSearchCV — búsqueda exhaustiva de hiperparámetros

```
Problema: ¿qué valores de max_depth y n_estimators funcionan mejor?

Sin GridSearch: probar manualmente combinación a combinación.
Con GridSearch: definir la rejilla, sklearn prueba TODAS las combinaciones
                y selecciona la mejor según CV.

Ejemplo: max_depth=[3,5,10] × n_estimators=[50,100]
  → 6 combinaciones × 5 folds = 30 entrenamientos

Resultado: .best_params_  → mejor combinación
           .best_score_   → score CV con esa combinación
           .best_estimator_ → modelo ya entrenado con todos los datos
```

In [ ]:
X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                            stratify=y, random_state=42)

# Definir rejilla de hiperparámetros
rejilla = {
    'max_depth':     [3, 5, 8, None],
    'n_estimators':  [50, 100, 200],
    'min_samples_split': [2, 5, 10],
}

gs = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid=rejilla,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring='accuracy',
    refit=True,     # re-entrena con la mejor combinación sobre todos los datos de train
    n_jobs=-1,
    verbose=1,
)
gs.fit(X_tr, y_tr)

print(f"\nMejores hiperparámetros:")
for k, v in gs.best_params_.items():
    print(f"  {k}: {v}")
print(f"\nScore CV (train): {gs.best_score_:.4f}")
print(f"Accuracy en test: {accuracy_score(y_te, gs.predict(X_te)):.4f}")

# Visualizar resultados como heatmap (n_estimators vs max_depth)
resultados_df = pd.DataFrame(gs.cv_results_)

# Filtrar a min_samples_split=2 para visualizar en 2D
sub = resultados_df[resultados_df['param_min_samples_split'] == 2].copy()
sub['max_depth_str'] = sub['param_max_depth'].apply(lambda x: str(x))

pivot = sub.pivot_table(
    values='mean_test_score',
    index='max_depth_str',
    columns='param_n_estimators'
)

fig, ax = plt.subplots(figsize=(7, 4))
import matplotlib
im = ax.imshow(pivot.values, cmap='YlGnBu', aspect='auto')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_xlabel('n_estimators'); ax.set_ylabel('max_depth')
ax.set_title('Accuracy CV (min_samples_split=2)')
plt.colorbar(im, ax=ax)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f'{pivot.values[i,j]:.3f}', ha='center', va='center', fontsize=9)
plt.tight_layout(); plt.show()

---
## 5. RandomizedSearchCV — búsqueda eficiente

```
Problema de GridSearch: crece exponencialmente con más hiperparámetros.
  3 params × 4 valores cada uno = 64 combinaciones
  5 params × 4 valores = 1024 combinaciones

RandomizedSearchCV: muestrea n_iter combinaciones al azar.
  → Mucho más rápido.
  → Casi tan bueno en la práctica (paper Bergstra & Bengio 2012).
  → Permite rangos continuos (scipy.stats.distributions).

Cuándo usar cada uno:
  GridSearch:    ≤3 hiperparámetros, rangos pequeños y discretos
  RandomSearch:  muchos hiperparámetros, rangos continuos, poco tiempo
```

In [ ]:
from scipy.stats import randint, uniform
import time

# Espacio de búsqueda con distribuciones continuas
espacio = {
    'n_estimators':       randint(50, 500),
    'max_depth':          [3, 5, 8, 10, None],
    'min_samples_split':  randint(2, 15),
    'min_samples_leaf':   randint(1, 8),
    'max_features':       ['sqrt', 'log2', 0.5, 0.7],
}

rs = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=espacio,
    n_iter=40,    # probar 40 combinaciones al azar
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring='accuracy',
    refit=True,
    n_jobs=-1,
    random_state=42,
    verbose=0,
)

t0 = time.time()
rs.fit(X_tr, y_tr)
t1 = time.time()

print(f"RandomizedSearchCV completado en {t1-t0:.1f}s")
print(f"\nMejores hiperparámetros:")
for k, v in rs.best_params_.items():
    print(f"  {k}: {v}")
print(f"\nScore CV (train): {rs.best_score_:.4f}")
print(f"Accuracy en test: {accuracy_score(y_te, rs.predict(X_te)):.4f}")

# Comparativa rápida
print(f"\n{'Método':25} {'Score CV':>10} {'Acc test':>10}")
print("-" * 48)
print(f"{'GridSearchCV':25} {gs.best_score_:>10.4f} {accuracy_score(y_te, gs.predict(X_te)):>10.4f}")
print(f"{'RandomizedSearchCV':25} {rs.best_score_:>10.4f} {accuracy_score(y_te, rs.predict(X_te)):>10.4f}")

---
## 6. Pipeline + GridSearchCV

```
El patrón más importante en ML de producción:
  Pipeline(steps) + GridSearchCV → búsqueda sin data leakage

Sintaxis para nombrar hiperparámetros en Pipeline:
  'nombrePaso__hiperparametro'

  Ejemplo:
    Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression())])
    param_grid = {'clf__C': [0.01, 0.1, 1, 10], 'clf__penalty': ['l1', 'l2']}

  El scaler se ajusta en train de cada fold y transforma test → SIN leakage.
```

In [ ]:
# Pipeline con LogisticRegression
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=5000, random_state=42)),
])

rejilla_pipe = {
    'clf__C':       [0.001, 0.01, 0.1, 1, 10, 100],
    'clf__penalty': ['l1', 'l2'],
    'clf__solver':  ['liblinear'],  # soporta l1 y l2
}

gs_pipe = GridSearchCV(
    pipe, rejilla_pipe,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring='accuracy',
    refit=True, n_jobs=-1
)
gs_pipe.fit(X_tr, y_tr)

print("Pipeline + GridSearchCV — Logistic Regression")
print(f"Mejores parámetros: {gs_pipe.best_params_}")
print(f"Score CV:           {gs_pipe.best_score_:.4f}")
print(f"Accuracy test:      {accuracy_score(y_te, gs_pipe.predict(X_te)):.4f}")

# Visualizar: C vs penalty
df_res = pd.DataFrame(gs_pipe.cv_results_)
pivot2 = df_res.pivot_table(
    values='mean_test_score',
    index='param_clf__penalty',
    columns='param_clf__C'
)

fig, ax = plt.subplots(figsize=(9, 3))
im2 = ax.imshow(pivot2.values, cmap='Blues', aspect='auto')
ax.set_xticks(range(len(pivot2.columns)))
ax.set_xticklabels(pivot2.columns)
ax.set_yticks(range(len(pivot2.index)))
ax.set_yticklabels(pivot2.index)
ax.set_xlabel('C'); ax.set_ylabel('penalty')
ax.set_title('Accuracy CV — Pipeline LogisticRegression')
plt.colorbar(im2, ax=ax)
for i in range(len(pivot2.index)):
    for j in range(len(pivot2.columns)):
        ax.text(j, i, f'{pivot2.values[i,j]:.3f}', ha='center', va='center', fontsize=9)
plt.tight_layout(); plt.show()

---
## Práctica

### Ejercicio 1 — Comparar CV vs holdout

**Enunciado:** Genera un dataset pequeño:

```python
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=150, n_features=10, n_informative=5, random_state=42)
```

1. Evalúa un DecisionTree depth=4 con 50 splits aleatorios (test_size=0.2)
2. Evalúa el mismo árbol con 5-Fold StratifiedCV, 10 repeticiones (usa `RepeatedStratifiedKFold`)
3. ¿Cuál estimación tiene menos varianza? ¿Por qué?
4. ¿Qué método confías más para elegir un modelo?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import (
    train_test_split, cross_val_score, RepeatedStratifiedKFold
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X, y = make_classification(n_samples=150, n_features=10, n_informative=5, random_state=42)
modelo = DecisionTreeClassifier(max_depth=4)

# 1. Holdout aleatorio
scores_holdout = []
for _ in range(50):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2)
    scores_holdout.append(accuracy_score(y_te, modelo.fit(X_tr, y_tr).predict(X_te)))

# 2. Repeated Stratified CV
rkf = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
scores_cv = cross_val_score(modelo, X, y, cv=rkf, scoring='accuracy', n_jobs=-1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(scores_holdout, bins=15, color='steelblue', alpha=0.7, edgecolor='white')
axes[0].axvline(np.mean(scores_holdout), color='red', lw=2, label=f'Media={np.mean(scores_holdout):.4f}')
axes[0].set_title(f'Holdout aleatorio (n=50)\nStd={np.std(scores_holdout):.4f}')
axes[0].legend()

axes[1].hist(scores_cv, bins=15, color='green', alpha=0.7, edgecolor='white')
axes[1].axvline(np.mean(scores_cv), color='red', lw=2, label=f'Media={np.mean(scores_cv):.4f}')
axes[1].set_title(f'Repeated StratifiedKFold (5×10)\nStd={np.std(scores_cv):.4f}')
axes[1].legend()

plt.tight_layout(); plt.show()

print(f"Holdout:  media={np.mean(scores_holdout):.4f}, std={np.std(scores_holdout):.4f}")
print(f"CV:       media={np.mean(scores_cv):.4f}, std={np.std(scores_cv):.4f}")
print(f"\nEl CV reduce la std. Confiar más en CV para selección de modelos.")
```
</details>

---
### Ejercicio 2 — GridSearch para un árbol de decisión

**Enunciado:** Usa el dataset de cáncer de mama:

1. Define un `GridSearchCV` para `DecisionTreeClassifier` explorando:
   - `max_depth`: [2, 3, 4, 5, 6, 8, None]
   - `min_samples_split`: [2, 5, 10, 20]
   - `criterion`: ['gini', 'entropy']
2. Muestra los mejores parámetros y el score en test
3. Compara el árbol por defecto (sin optimizar) vs el optimizado
4. Visualiza el heatmap de scores para criterion=gini

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

rejilla = {
    'max_depth':         [2, 3, 4, 5, 6, 8, None],
    'min_samples_split': [2, 5, 10, 20],
    'criterion':         ['gini', 'entropy'],
}

gs = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    rejilla,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring='accuracy', refit=True, n_jobs=-1
)
gs.fit(X_tr, y_tr)

print(f"Mejores parámetros: {gs.best_params_}")
print(f"Score CV: {gs.best_score_:.4f}")

# Comparativa
arbol_default = DecisionTreeClassifier(random_state=42).fit(X_tr, y_tr)
print(f"\n{'Modelo':25} {'Acc test':>10}")
print("-" * 38)
print(f"{'Árbol por defecto':25} {accuracy_score(y_te, arbol_default.predict(X_te)):>10.4f}")
print(f"{'Árbol optimizado':25} {accuracy_score(y_te, gs.predict(X_te)):>10.4f}")

# Heatmap
df_res = pd.DataFrame(gs.cv_results_)
sub = df_res[df_res['param_criterion'] == 'gini'].copy()
sub['depth_str'] = sub['param_max_depth'].apply(lambda x: str(x))
pivot = sub.pivot_table('mean_test_score', 'depth_str', 'param_min_samples_split')

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(pivot.values, cmap='YlGnBu', aspect='auto')
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
ax.set_xlabel('min_samples_split'); ax.set_ylabel('max_depth')
ax.set_title('Accuracy CV (criterion=gini)')
plt.colorbar(im, ax=ax)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f'{pivot.values[i,j]:.3f}', ha='center', va='center', fontsize=9)
plt.tight_layout(); plt.show()
```
</details>

---
### Ejercicio 3 — Pipeline completo con búsqueda de hiperparámetros

**Enunciado:** Crea un pipeline que incluya escalado + Random Forest y busca sus mejores hiperparámetros:

```python
from sklearn.datasets import load_wine
X, y = load_wine(return_X_y=True)
```

1. Pipeline: `StandardScaler → RandomForestClassifier`
2. Usa `RandomizedSearchCV` para explorar:
   - `clf__n_estimators`: randint(50, 300)
   - `clf__max_depth`: [3, 5, None]
   - `clf__max_features`: ['sqrt', 'log2']
   - `clf__min_samples_leaf`: randint(1, 6)
3. n_iter=30, cv=5
4. Muestra mejores parámetros, score CV y accuracy en test

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
from scipy.stats import randint
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X, y = load_wine(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    RandomForestClassifier(random_state=42, n_jobs=-1)),
])

espacio = {
    'clf__n_estimators':   randint(50, 300),
    'clf__max_depth':      [3, 5, None],
    'clf__max_features':   ['sqrt', 'log2'],
    'clf__min_samples_leaf': randint(1, 6),
}

rs = RandomizedSearchCV(
    pipe, espacio, n_iter=30,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring='accuracy', refit=True, n_jobs=-1, random_state=42
)
rs.fit(X_tr, y_tr)

print("Mejores hiperparámetros:")
for k, v in rs.best_params_.items():
    print(f"  {k}: {v}")
print(f"\nScore CV:      {rs.best_score_:.4f}")
print(f"Accuracy test: {accuracy_score(y_te, rs.predict(X_te)):.4f}")
```
</details>

---
### Ejercicio 4 — Múltiples métricas de evaluación

**Enunciado:** Queremos evaluar un clasificador no solo por accuracy sino también por F1:

```python
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=500, n_features=15, n_informative=8,
                           weights=[0.7, 0.3], random_state=42)  # clases desbalanceadas
```

1. Evalúa LogisticRegression con `cross_validate` usando accuracy, F1 y AUC
2. ¿Cuál métrica cambia más entre folds?
3. ¿Cambiaría tu decisión de usar accuracy vs F1 en este caso?

*Pista: usa `from sklearn.model_selection import cross_validate`*

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.linear_model import LogisticRegression

X, y = make_classification(n_samples=500, n_features=15, n_informative=8,
                           weights=[0.7, 0.3], random_state=42)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
m  = LogisticRegression(max_iter=5000, random_state=42)

resultados = cross_validate(
    m, X, y, cv=cv,
    scoring={'accuracy': 'accuracy', 'f1': 'f1', 'roc_auc': 'roc_auc'},
    return_train_score=False, n_jobs=-1
)

print(f"{'Métrica':12} {'Media':>8} {'Std':>8} {'Min':>8} {'Max':>8}")
print("-" * 48)
for k in ['accuracy', 'f1', 'roc_auc']:
    sc = resultados[f'test_{k}']
    print(f"{k:12} {sc.mean():>8.4f} {sc.std():>8.4f} {sc.min():>8.4f} {sc.max():>8.4f}")

print("\nConclusion:")
print("  Con clases desbalanceadas (70/30), la accuracy puede ser engañosa.")
print("  F1 y AUC dan una imagen más honesta del rendimiento real.")
print("  Usa F1 cuando los falsos negativos y falsos positivos tienen coste.")
```
</details>

---
### Ejercicio 5 (reto) — Comparador automático de modelos

**Enunciado:** Implementa `comparar_modelos(modelos_dict, X, y, cv=5)` que:

1. Evalúa cada modelo con StratifiedKFold y calcula accuracy, F1 y AUC
2. Devuelve un `pd.DataFrame` ordenado por AUC descendente
3. Plotea un gráfico de barras con error bars (std) para las 3 métricas
4. Destaca el mejor modelo en el gráfico

Pruébalo con: LogisticRegression, DecisionTree depth=5, RandomForest 100.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.datasets import load_breast_cancer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

def comparar_modelos(modelos_dict, X, y, cv=5):
    cv_obj = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    metricas = {'accuracy': 'accuracy', 'f1': 'f1', 'roc_auc': 'roc_auc'}
    filas = []
    for nombre, m in modelos_dict.items():
        res = cross_validate(m, X, y, cv=cv_obj, scoring=metricas, n_jobs=-1)
        fila = {'modelo': nombre}
        for metrica in metricas:
            sc = res[f'test_{metrica}']
            fila[f'{metrica}_media'] = sc.mean()
            fila[f'{metrica}_std']   = sc.std()
        filas.append(fila)

    df = pd.DataFrame(filas).sort_values('roc_auc_media', ascending=False)

    # Gráfico
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    colores = ['steelblue', 'green', 'coral']
    mejor = df.iloc[0]['modelo']

    for ax, met, color in zip(axes, ['accuracy', 'f1', 'roc_auc'], colores):
        medias = df[f'{met}_media'].values
        stds   = df[f'{met}_std'].values
        nombres = df['modelo'].values
        barras = ax.bar(nombres, medias, yerr=stds, capsize=5,
                        color=[color if n != mejor else 'gold' for n in nombres],
                        edgecolor='gray')
        ax.set_ylim(0.7, 1.05)
        ax.set_title(met.upper())
        ax.tick_params(axis='x', rotation=15)
        ax.grid(True, axis='y', alpha=0.3)
        for barra, media, std in zip(barras, medias, stds):
            ax.text(barra.get_x() + barra.get_width()/2, media + std + 0.005,
                    f'{media:.3f}', ha='center', va='bottom', fontsize=9)

    plt.suptitle(f'Comparativa de modelos — Mejor: {mejor}', fontsize=11)
    plt.tight_layout(); plt.show()
    return df

# Prueba
X, y = load_breast_cancer(return_X_y=True)
modelos = {
    'LogisticReg':  LogisticRegression(max_iter=5000, random_state=42),
    'Árbol depth=5': DecisionTreeClassifier(max_depth=5, random_state=42),
    'RandomForest': RandomForestClassifier(100, random_state=42, n_jobs=-1),
}
tabla = comparar_modelos(modelos, X, y, cv=5)
print(tabla[['modelo','accuracy_media','f1_media','roc_auc_media']].to_string(index=False))
```
</details>

---
## Resumen

```python
from sklearn.model_selection import (
    StratifiedKFold, cross_val_score,
    GridSearchCV, RandomizedSearchCV
)
from sklearn.pipeline import Pipeline

# Evaluación fiable
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(modelo, X, y, cv=cv, scoring='accuracy')
print(f"{scores.mean():.4f} ± {scores.std():.4f}")

# Búsqueda de hiperparámetros (exhaustiva)
gs = GridSearchCV(modelo, param_grid={'max_depth': [3,5,None]}, cv=cv, refit=True)
gs.fit(X_tr, y_tr)
print(gs.best_params_, gs.best_score_)

# Búsqueda de hiperparámetros (eficiente)
rs = RandomizedSearchCV(modelo, param_distributions=espacio, n_iter=40, cv=cv, refit=True)

# Pipeline + GridSearch (sin data leakage)
pipe = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression())])
gs_pipe = GridSearchCV(pipe, {'clf__C': [0.1, 1, 10]}, cv=cv)
gs_pipe.fit(X_tr, y_tr)
```

## ¿Qué viene en la próxima clase?
En la **Clase 8** aprenderemos **Clustering**: KMeans, Elbow method y Silhouette score para encontrar grupos en datos sin etiquetas.